# 16 — Distributed Runtime on Kind (Real gRPC)

This notebook demonstrates the **GrpcRuntime** running on real Kubernetes pods.
Two runtime nodes are deployed in the Kind cluster, each serving different agents.
The notebook acts as a **client** and dispatches messages to remote agents via gRPC.

### Runtime Hierarchy

```
BaseRuntime (ABC, core/runtime/)
├── LocalRuntime (core/runtime/) — in-process, asyncio
└── BaseRemoteRuntime (integrations/runtime/) — adds _remote_send
    ├── GrpcRuntime — gRPC unary stubs  ← this notebook
    └── RestateRuntime — Restate ingress HTTP
```

### Kind Cluster Layout

| Pod | Agents | K8s Service | Port-forward |
|-----|--------|-------------|-------------|
| runtime-node-a | echo, greeter | runtime-node-a.af-runtime.svc:50051 | localhost:50051 |
| runtime-node-b | summarizer, translator | runtime-node-b.af-runtime.svc:50051 | localhost:50052 |


In [1]:
import subprocess
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

## 1. Inheritance Hierarchy — Proof

All runtimes share a common `BaseRuntime` ABC in `core/runtime/`.
Handler registration, topic subscription, and introspection are
defined once and inherited everywhere.

In [2]:
from raavan.core.runtime import BaseRuntime, LocalRuntime, AgentRuntime
from raavan.integrations.runtime import BaseRemoteRuntime
from raavan.integrations.runtime.grpc import GrpcRuntime
from raavan.integrations.runtime.restate import RestateRuntime

print('=== Inheritance checks ===')
print(f'LocalRuntime   -> BaseRuntime: {issubclass(LocalRuntime, BaseRuntime)}')
print(f'GrpcRuntime    -> BaseRuntime: {issubclass(GrpcRuntime, BaseRuntime)}')
print(f'RestateRuntime -> BaseRuntime: {issubclass(RestateRuntime, BaseRuntime)}')
print()
print(f'GrpcRuntime    -> BaseRemoteRuntime: {issubclass(GrpcRuntime, BaseRemoteRuntime)}')
print(f'RestateRuntime -> BaseRemoteRuntime: {issubclass(RestateRuntime, BaseRemoteRuntime)}')
print(f'LocalRuntime   -> BaseRemoteRuntime: {issubclass(LocalRuntime, BaseRemoteRuntime)}')
print()
print('=== MRO ===')
for cls in [LocalRuntime, GrpcRuntime, RestateRuntime]:
    chain = ' -> '.join(c.__name__ for c in cls.__mro__)
    print(f'{cls.__name__}: {chain}')

=== Inheritance checks ===
LocalRuntime   -> BaseRuntime: True
GrpcRuntime    -> BaseRuntime: True
RestateRuntime -> BaseRuntime: True

GrpcRuntime    -> BaseRemoteRuntime: True
RestateRuntime -> BaseRemoteRuntime: True
LocalRuntime   -> BaseRemoteRuntime: False

=== MRO ===
LocalRuntime: LocalRuntime -> BaseRuntime -> ABC -> object
GrpcRuntime: GrpcRuntime -> BaseRemoteRuntime -> BaseRuntime -> ABC -> object
RestateRuntime: RestateRuntime -> BaseRemoteRuntime -> BaseRuntime -> ABC -> object


## 2. Verify Runtime Nodes in Kind

Check that both pods are running and serving gRPC.

In [3]:
result = subprocess.run(
    ['kubectl', 'get', 'pods', '-n', 'af-runtime', '-l', 'component=grpc-runtime',
     '-o', 'custom-columns=NAME:.metadata.name,STATUS:.status.phase,IP:.status.podIP',
     '--no-headers'],
    capture_output=True, text=True,
)
print('=== Runtime Nodes in Kind ===')
print(result.stdout.strip())
assert 'Running' in result.stdout, 'Runtime pods not running!'
print('\n\u2713 Both nodes are running')

=== Runtime Nodes in Kind ===
runtime-node-a-988cf8c4b-dt7q2   Running   10.244.0.32
runtime-node-b-c8546976c-95dtf   Running   10.244.0.31

✓ Both nodes are running


## 3. Create Client Runtime

The notebook creates a `GrpcRuntime` configured as a **client only** —
no local handlers, all dispatches go to Kind pods via `remote_peers`.

In [4]:
from raavan.core.runtime import AgentId, MessageContext
from typing import Any

client_rt = GrpcRuntime(
    listen_address='0.0.0.0:50099',
    remote_peers={
        'echo':       'localhost:50051',   # Node A
        'greeter':    'localhost:50051',   # Node A
        'summarizer': 'localhost:50052',   # Node B
        'translator': 'localhost:50052',   # Node B
    },
)
client_rt._started = True  # client-only, no server needed

print(f'Remote peers ({len(client_rt._remote_peers)}):')
for agent_type, addr in client_rt._remote_peers.items():
    node = 'A' if '50051' in addr else 'B'
    print(f'  {agent_type:12s} -> {addr}  (Node {node})')

Remote peers (4):
  echo         -> localhost:50051  (Node A)
  greeter      -> localhost:50051  (Node A)
  summarizer   -> localhost:50052  (Node B)
  translator   -> localhost:50052  (Node B)


## 4. Echo Agent (Node A) — Cross-pod gRPC

In [5]:
result = await client_rt.send_message(
    {'message': 'Hello from the notebook!', 'timestamp': '2026-04-02'},
    sender=AgentId('notebook', 'user'),
    recipient=AgentId('echo', 'test-1'),
)
print('=== Echo (Node A) ===')
for k, v in result.items():
    print(f'  {k}: {v}')

=== Echo (Node A) ===
  agent: echo
  node_id: echo/test-1
  echo: {'message': 'Hello from the notebook!', 'timestamp': '2026-04-02'}
  runtime: GrpcRuntime


## 5. Greeter Agent (Node A)

In [6]:
result = await client_rt.send_message(
    {'name': 'Kind cluster'},
    sender=AgentId('notebook', 'user'),
    recipient=AgentId('greeter', 'demo-1'),
)
print('=== Greeter (Node A) ===')
for k, v in result.items():
    print(f'  {k}: {v}')

=== Greeter (Node A) ===
  agent: greeter
  node_id: greeter/demo-1
  greeting: Hello, Kind cluster! Greetings from greeter/demo-1.


## 6. Summarizer Agent (Node B)

In [7]:
long_text = (
    'The distributed runtime architecture allows agents to run on separate '
    'pods in a Kubernetes cluster. Each pod hosts a GrpcRuntime node that '
    'serves one or more agent types. Cross-pod communication happens via '
    'gRPC unary calls with JSON-encoded envelopes. The BaseRuntime ABC in '
    'core/ provides shared handler registration, while BaseRemoteRuntime '
    'in integrations/ adds lifecycle guards and local/remote dispatch logic.'
)

result = await client_rt.send_message(
    {'text': long_text},
    sender=AgentId('notebook', 'user'),
    recipient=AgentId('summarizer', 'doc-1'),
)
print('=== Summarizer (Node B) ===')
for k, v in result.items():
    print(f'  {k}: {v}')

=== Summarizer (Node B) ===
  agent: summarizer
  node_id: summarizer/doc-1
  summary: The distributed runtime architecture allows agents to run on separate pods in a ...
  original_length: 415


## 7. Translator Agent (Node B)

In [8]:
result = await client_rt.send_message(
    {'text': 'Hello distributed world'},
    sender=AgentId('notebook', 'user'),
    recipient=AgentId('translator', 'demo-1'),
)
print('=== Translator (Node B) ===')
for k, v in result.items():
    print(f'  {k}: {v}')

=== Translator (Node B) ===
  agent: translator
  node_id: translator/demo-1
  translation: dlrow detubirtsid olleH
  source_lang: en
  target_lang: reverse


## 8. Multi-Node Pipeline — greeter (A) → summarizer (B) → translator (B)

Chain three agents across two pods. Each step feeds its output
into the next agent, crossing pod boundaries via gRPC.

In [9]:
# Step 1: greeter on Node A
step1 = await client_rt.send_message(
    {'name': 'Kubernetes'},
    sender=AgentId('notebook', 'pipeline'),
    recipient=AgentId('greeter', 'step-1'),
)
print(f'Step 1 (greeter @ Node A):    {step1["greeting"]}')

# Step 2: summarizer on Node B
step2 = await client_rt.send_message(
    {'text': step1['greeting']},
    sender=AgentId('greeter', 'step-1'),
    recipient=AgentId('summarizer', 'step-2'),
)
print(f'Step 2 (summarizer @ Node B): {step2["summary"]}')

# Step 3: translator on Node B
step3 = await client_rt.send_message(
    {'text': step2['summary']},
    sender=AgentId('summarizer', 'step-2'),
    recipient=AgentId('translator', 'step-3'),
)
print(f'Step 3 (translator @ Node B): {step3["translation"]}')

print(f'\n\u2713 Pipeline crossed 2 pod boundaries via gRPC')

Step 1 (greeter @ Node A):    Hello, Kubernetes! Greetings from greeter/step-1.
Step 2 (summarizer @ Node B): Hello, Kubernetes! Greetings from greeter/step-1.
Step 3 (translator @ Node B): .1-pets/reteerg morf sgniteerG !setenrebuK ,olleH

✓ Pipeline crossed 2 pod boundaries via gRPC


## 9. Round-Robin — All 4 Agents Across Both Nodes

In [10]:
agents = [
    ('echo',       {'ping': 'pong'}),
    ('greeter',    {'name': 'Kind cluster'}),
    ('summarizer', {'text': 'A quick brown fox jumps over the lazy dog near the riverbank'}),
    ('translator', {'text': 'Hello distributed world'}),
]

print('=== Round-robin: all 4 agents across 2 nodes ===\n')
for agent_type, payload in agents:
    result = await client_rt.send_message(
        payload,
        sender=AgentId('notebook', 'roundrobin'),
        recipient=AgentId(agent_type, 'rr-1'),
    )
    node = 'A' if agent_type in ('echo', 'greeter') else 'B'
    print(f'  [{agent_type:12s}] Node {node} -> {result}')

print(f'\n\u2713 All 4 agents responded from Kind pods')

=== Round-robin: all 4 agents across 2 nodes ===

  [echo        ] Node A -> {'agent': 'echo', 'node_id': 'echo/rr-1', 'echo': {'ping': 'pong'}, 'runtime': 'GrpcRuntime'}
  [greeter     ] Node A -> {'agent': 'greeter', 'node_id': 'greeter/rr-1', 'greeting': 'Hello, Kind cluster! Greetings from greeter/rr-1.'}
  [summarizer  ] Node B -> {'agent': 'summarizer', 'node_id': 'summarizer/rr-1', 'summary': 'A quick brown fox jumps over the lazy dog near the riverbank', 'original_length': 60}
  [translator  ] Node B -> {'agent': 'translator', 'node_id': 'translator/rr-1', 'translation': 'dlrow detubirtsid olleH', 'source_lang': 'en', 'target_lang': 'reverse'}

✓ All 4 agents responded from Kind pods


## 10. Verify Pod Logs

In [11]:
for node in ['a', 'b']:
    result = subprocess.run(
        ['kubectl', 'logs', f'deployment/runtime-node-{node}',
         '-n', 'af-runtime', '--tail=5'],
        capture_output=True, text=True,
    )
    print(f'=== Node {node.upper()} (last 5 log lines) ===')
    print(result.stdout.strip())
    print()

print('\u2713 Distributed runtime test complete \u2014 real gRPC over Kind!')

=== Node A (last 5 log lines) ===
{"timestamp": "2026-04-02T15:44:38.217871+00:00", "level": "INFO", "name": "raavan.integrations.runtime.grpc.runtime", "message": "GrpcRuntime started (listen=0.0.0.0:50051, peers=0, handlers=['echo', 'greeter'])"}
{"timestamp": "2026-04-02T15:44:38.217992+00:00", "level": "INFO", "name": "raavan.runtime.node", "message": "Node ready â€” listening on 0.0.0.0:50051, serving: ['echo', 'greeter']"}

=== Node B (last 5 log lines) ===
{"timestamp": "2026-04-02T15:44:38.173581+00:00", "level": "INFO", "name": "raavan.integrations.runtime.grpc.runtime", "message": "GrpcRuntime started (listen=0.0.0.0:50051, peers=0, handlers=['summarizer', 'translator'])"}
{"timestamp": "2026-04-02T15:44:38.173746+00:00", "level": "INFO", "name": "raavan.runtime.node", "message": "Node ready â€” listening on 0.0.0.0:50051, serving: ['summarizer', 'translator']"}

✓ Distributed runtime test complete — real gRPC over Kind!


## Summary

```
Notebook (client GrpcRuntime)
    │
    ├── gRPC → Node A (localhost:50051 → pod runtime-node-a)
    │              ├── echo agent
    │              └── greeter agent
    │
    └── gRPC → Node B (localhost:50052 → pod runtime-node-b)
                   ├── summarizer agent
                   └── translator agent
```

**Key takeaways:**
1. Same `GrpcRuntime` class used as both **server** (in pods) and **client** (in notebook)
2. `BaseRuntime` hierarchy means `register()` / `send_message()` work identically
3. Handler code is completely runtime-agnostic — no gRPC imports, no transport awareness
4. Cross-pod pipeline: agents chain across pod boundaries transparently